## Gold Layer: Business Intelligence & KPI Aggregates

In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
import dlt
from pyspark.sql.functions import col, sum, count, avg, count_distinct, round, current_timestamp, lit, broadcast

# Optimized for high-volume 110M row shuffles to prevent data skew
spark.conf.set("spark.sql.shuffle.partitions", "400")

def add_audit_metadata(df):
    """Appends traceability metadata to meet Domain 5 governance standards."""
    return df.withColumn("ingestion_timestamp", current_timestamp()) \
             .withColumn("processed_by_user", lit("dlt_pipeline_service"))

## KPI 1: Product & Brand Performance Analysis
This table joins the validated `fact_sales` with the `dim_products_scd2` dimension. By leveraging a broadcast join, we calculate total revenue, units sold, and session counts per brand while maintaining point-in-time accuracy.

In [0]:
@dlt.table(
    name="product_performance",
    comment="Gold Layer: KPI from Sales and SCD2 Dimensions",
    table_properties={"quality": "gold"}
)
def calculate_product_performance():
    # Utilizing direct DLT references to establish automatic lineage arrows in the graph
    fact_sales = dlt.read("fact_sales") 
    dim_products = dlt.read("dim_products_scd2")

    # High-efficiency broadcast join to minimize network traffic during 110M row processing
    joined_df = fact_sales.join(broadcast(dim_products), on="product_id", how="inner")
    
    return add_audit_metadata(
        joined_df.groupBy(dim_products["brand"])
        .agg(
            round(sum(fact_sales["price"]), 2).alias("actual_revenue"),
            round(avg(dim_products["price"]), 2).alias("historical_catalog_price"),
            count(fact_sales["product_id"]).alias("total_units_sold"),
            count_distinct(fact_sales["user_session"]).alias("total_sessions")
        )
    )

## KPI 2: Customer Behavioral Analytics
This table tracks user interaction patterns by aggregating session counts and interaction values directly from the Silver layer. This provides the Marketing team with a granular view of user engagement across the 110M event stream.

In [0]:
@dlt.table(
    name="customer_metrics",
    comment="Gold Layer: behavioral conversion behavior",
    table_properties={"quality": "gold"}
)
def customer_journey_metrics():
    # Direct reference to the cleaned event stream for behavioral modeling
    return add_audit_metadata(
        dlt.read("events_cleaned")
        .groupBy("user_id")
        .agg(
            count_distinct("user_session").alias("total_sessions"),
            count(col("event_type")).alias("total_interactions"),
            round(avg(col("price")), 2).alias("avg_interaction_value")
        )
    )